<a href="https://colab.research.google.com/github/OrastaRakhmatullayeva/AI-projects/blob/main/Hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring NLP with Hugging Face Transformers

**Orasta Rahmatullayeva** · Python · PyTorch · Hugging Face

I explore pretrained NLP models, dive into tokenization and model internals, and fine-tune DistilBERT for movie review sentiment classification.

**Result:** 90% accuracy on a 500-review IMDb evaluation subset.

---

## 1. Environment setup

Install dependencies and check GPU availability.


In [ ]:
!pip install -q -U transformers datasets
import torch, transformers
print(transformers.__version__)
print(torch.cuda.is_available())

5.22.1
True


### Hugging Face authentication

Connect to the Hub using notebook login or a Colab secret (`HF_corp`).


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_corp'))

## 2. Pretrained NLP pipelines

Start with sentiment analysis using a pretrained DistilBERT model.


In [ ]:
from transformers import pipeline

MODEL = "distilbert-base-uncased-finetuned-sst-2-english"

clf=pipeline("sentiment-analysis",model=MODEL)

print(clf("I love this phone"))
print(clf(["Great quality",'It arrived broken']))

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9997873902320862}]
[{'label': 'POSITIVE', 'score': 0.9998776912689209}, {'label': 'NEGATIVE', 'score': 0.9991670846939087}]


### Beyond sentiment analysis

Explore named entity recognition (BERT), zero-shot classification (BART), and text generation (GPT-2).


In [ ]:
from transformers import pipeline

ner = pipeline("token-classification",
               model="dslim/bert-base-NER",
               aggregation_strategy="simple")
print(ner("Elon Musk works at Tesla in Texas"))

zs = pipeline("zero-shot-classification",
              model="facebook/bart-large-mnli")
print(zs("The battery dies in one hour",
         candidate_labels=["battery", "price", "delivery"]))

gen = pipeline("text-generation", model="gpt2")
print(gen("Machine learning is", max_new_tokens=20))


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

... (19 outputs left out)

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[{'generated_text': "Machine learning is a very complex and unpredictable process. It's easy to lose focus.\n\nYour brain is like"}]


### Understanding predictions

Compare positive and negative labels with their confidence scores. Setting `top_k=None` returns scores for all labels.


In [ ]:
from transformers import pipeline

MODEL = "distilbert-base-uncased-finetuned-sst-2-english"

clf=pipeline("sentiment-analysis",model=MODEL)

print(clf("The battery is low"))
print(clf(["Good search",'It is worth to watch','Ever never']))

print(clf("I love it",top_k=None)) #bor labelarni chiqaradi, bizda positive and negative labellar bor

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9996966123580933}]
[{'label': 'POSITIVE', 'score': 0.9998595714569092}, {'label': 'POSITIVE', 'score': 0.9998766183853149}, {'label': 'NEGATIVE', 'score': 0.6943067908287048}]
[{'label': 'POSITIVE', 'score': 0.9998799562454224}, {'label': 'NEGATIVE', 'score': 0.00012001476716250181}]


## 3. Tokenization and input preparation

Inspect subword tokens, token IDs, and decoded text to understand how a model reads language.


In [ ]:
from transformers import AutoTokenizer

MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

text = 'Tokenization is amaaaazing!'
enc = tok(text)
print(list(enc.keys()))
print(tok.tokenize(text))
print(enc["input_ids"])
print(tok.decode(enc["input_ids"]))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

['input_ids', 'token_type_ids', 'attention_mask']
['token', '##ization', 'is', 'ama', '##aa', '##azi', '##ng', '!']
[101, 19204, 3989, 2003, 25933, 11057, 16103, 3070, 999, 102]
[CLS] tokenization is amaaaazing! [SEP]


### Padding, truncation, and attention masks

Prepare batches of different-length sentences. Attention masks distinguish meaningful tokens from padding.


In [ ]:
from transformers import AutoTokenizer

tok=AutoTokenizer.from_pretrained('distilbert-base-uncased')

texts=['I love this phone',
       'The battery life lasts less and i want my money back']

batch=tok(texts,padding='max_length',truncation=True,
          max_length=16,return_tensors='pt')

print(batch['input_ids'].shape)
print(batch['attention_mask'])

torch.Size([2, 16])
tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])


In [ ]:

tok=AutoTokenizer.from_pretrained('distilbert-base-uncased')

texts=['Unbelievable quality!']
batch=tok(texts,padding=True,return_tensors='pt')

print(batch['input_ids'].shape)
enc = tok(texts)
print(list(enc.keys()))
print(tok.tokenize(texts))

texts=['Unbelievable quality!','Hello','You are my destiny']

batch=tok(texts,padding=True,return_tensors='pt')

print(batch['input_ids'].shape)


batch=tok(texts,padding='max_length',max_length=4,truncation=True,return_tensors='pt')

print(batch['input_ids'].shape)


torch.Size([1, 5])
['input_ids', 'token_type_ids', 'attention_mask']
['unbelievable', 'quality', '!']
torch.Size([3, 6])
torch.Size([3, 4])


## 4. Under the hood

Run inference directly with PyTorch, convert logits into probabilities, and inspect the label mapping.


In [ ]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
model.eval()

texts = ["I love this phone", "The battery is terrible"]
batch = tok(texts, padding=True, return_tensors="pt")
with torch.no_grad():
    out = model(**batch)

probs = out.logits.softmax(dim=-1)
print(out.logits, probs.sum(dim=-1))
print(model.config.id2label)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[-4.0942,  4.3616],
        [ 4.3241, -3.5237]]) tensor([1., 1.])
{0: 'NEGATIVE', 1: 'POSITIVE'}


### Model architecture

Inspect DistilBERT's configuration, parameter count, and hidden-state dimensions. The base model has 6 layers, 12 attention heads, and a hidden size of 768.


In [ ]:
from transformers import AutoTokenizer,AutoConfig,AutoModel

MODEL='distilbert-base-uncased-finetuned-sst-2-english'
cfg=AutoConfig.from_pretrained(MODEL)

layers,heads=cfg.num_hidden_layers,cfg.num_attention_heads
print(cfg.hidden_size,layers,heads)

model=AutoModel.from_pretrained(MODEL)
print(sum(p.numel() for p in model.parameters()))

tok = AutoTokenizer.from_pretrained(MODEL)

texts = ["I love this phone", "The battery is terrible"]
batch = tok(texts, padding=True, return_tensors="pt")
print(model(**batch).last_hidden_state.shape)
# model = AutoModelForSequenceClassification.from_pretrained(MODEL)
# model.eval()


768 6 12


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.weight | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


66362880
torch.Size([2, 6, 768])


## 5. Fine-tuning DistilBERT on IMDb

Use 2,000 training reviews and 500 evaluation reviews. Tokenize the text and set up accuracy as the evaluation metric.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments, set_seed

set_seed(0)
MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)
collator = DataCollatorWithPadding(tok)

ds = load_dataset("stanfordnlp/imdb")
train = ds["train"].shuffle(seed=0).select(range(2000))
test = ds["test"].shuffle(seed=0).select(range(500))

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=256)

def metrics(p):
    pred = p.predictions.argmax(-1)
    return {"accuracy": float((pred == p.label_ids).mean())}

train = train.map(tokenize, batched=True)
test = test.map(tokenize, batched=True)
train


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

... (3 outputs left out)

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})

### Training and evaluation

Fine-tune a binary classifier for two epochs using the Hugging Face Trainer.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2, id2label={0: "NEGATIVE", 1: "POSITIVE"})

args = TrainingArguments(
    output_dir="stanfordnlp/imdb-model", num_train_epochs=2,
    learning_rate=2e-5, per_device_train_batch_size=16,
    eval_strategy="epoch", report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=train,
                  eval_dataset=test, data_collator=collator,
                  compute_metrics=metrics)
trainer.train()
print(trainer.evaluate()["eval_accuracy"])

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.312395,0.878000
2,No log,0.259746,0.900000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
No log,0.259746,2,0.900000


0.9


### Save and test the model

Save the fine-tuned model and tokenizer locally, then use a classification pipeline for a new review.


In [ ]:
from transformers import pipeline

trainer.save_model("stanfordnlp/imdb-model")
tok.save_pretrained("stanfordnlp/imdb-model")

clf = pipeline("text-classification", model="stanfordnlp/imdb-model")
print(clf("it was boring, but I love it!"))
# print(clf(["Boring and too long", "I loved every minute"]))
# Hub ga yuklash (Write token kerak):
# trainer.push_to_hub("stanfordnlp/imdb-distilbert")

---

## Key takeaway

The fine-tuned model reached **90% accuracy** on the selected IMDb evaluation subset. This experiment helped me connect NLP pipelines with the underlying model architecture and the end-to-end fine-tuning workflow.
